# 03 · Innings table — one row per match-innings, proved against `matches`

**Goal:** aggregate the deliveries into an innings table (runs, extras by type, balls, wickets), prove the aggregation lost and duplicated nothing, then reconcile it with the recorded results: wicket margins, D/L margins, super overs and the seven-ball overs left open in notebook 01.

**Rules:** raw team strings stay raw (cleaning is a later pass); only the *aggregation* is under test. Super-over innings (3 and up) are flagged with `is_super_over`, never dropped, and never mixed into the totals used for results.

**Contents:** Part B build and prove the grain · Part C reconcile with `matches` · what this changes in notebook 01's worklist

## Part B — Build the innings table and prove its grain

### B1 · Setup

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd

from ipl_eda import innings, tables

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Works whether Jupyter starts in notebooks/ (the default) or in the repo root.
RAW = next((p for p in [Path("../data/procured/cricsheet"), Path("data/procured/cricsheet")] if p.exists()), None)
if RAW is None:
    raise FileNotFoundError("data/procured/cricsheet not found - see README.md")
PROCESSED = RAW.parents[1] / "processed"

columns = ["match_id", "innings", "ball", "batting_team", "bowling_team", "runs_off_bat", "extras",
           "wides", "noballs", "byes", "legbyes", "penalty", "wicket_type"]
deliveries = pd.read_csv(RAW / "all_matches.csv", usecols=columns, low_memory=False)
matches = tables.read_matches(PROCESSED / "matches.csv")        # the typed loader: no default pd.read_csv from here on
print(f"deliveries: {len(deliveries):,} rows | matches: {len(matches):,} rows")

deliveries: 295,732 rows | matches: 1,243 rows


### B2 · Build it

`innings.build_innings()` is a named aggregation on `groupby(["match_id", "innings"])`. Legal balls are the rows that are neither wides nor no-balls — not `actual_delivery`, which repeats across a wide. `is_super_over` is `innings > 2` (the Cricsheet README's rule). `wickets` follows the definition in `innings.NOT_WICKETS`, which section C1 justifies from the recorded results.

In [2]:
innings_table = innings.build_innings(deliveries)
print(f"{len(innings_table):,} rows x {innings_table.shape[1]} columns")
innings_table.head(6)

2,514 rows x 16 columns


,match_id,innings,batting_team,bowling_team,is_super_over,runs,runs_off_bat,extras,wides,noballs,byes,legbyes,penalty,deliveries,legal_balls,wickets
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,False,222,205,17,9,0,4,4,0,124,120,3
1,335982,2,Royal Challengers Bangalore,Kolkata Knight Riders,False,82,63,19,11,0,0,8,0,101,91,10
2,335983,1,Chennai Super Kings,Kings XI Punjab,False,240,234,6,3,1,0,2,0,124,120,5
3,335983,2,Kings XI Punjab,Chennai Super Kings,False,207,196,11,5,0,2,4,0,124,120,4
4,335984,1,Rajasthan Royals,Delhi Daredevils,False,129,122,7,3,0,1,3,0,122,120,8
5,335984,2,Delhi Daredevils,Rajasthan Royals,False,132,122,10,10,0,0,0,0,97,91,1


### B3 · Prove the grain

A total can be right while the grain is wrong (two innings merged, one split in two), so check the grain itself:

In [3]:
per_innings = deliveries.groupby(["match_id", "innings"])
numbers = innings_table.groupby("match_id")["innings"].agg(["min", "max", "nunique"])

print("(match_id, innings) pairs that occur more than once:", int(innings_table.duplicated(["match_id", "innings"]).sum()))
print("innings with more than one batting team in the deliveries:", int((per_innings["batting_team"].nunique() > 1).sum()))
print("matches whose innings numbers are not 1..k without gaps:", int((~((numbers["min"] == 1) & (numbers["max"] == numbers["nunique"]))).sum()))
print("matches by number of innings:", numbers["nunique"].value_counts().sort_index().to_dict(),
      "| rows:", len(innings_table), "| super-over innings:", int(innings_table["is_super_over"].sum()))

(match_id, innings) pairs that occur more than once: 0
innings with more than one batting team in the deliveries: 0
matches whose innings numbers are not 1..k without gaps: 0
matches by number of innings: {1: 6, 2: 1221, 4: 15, 6: 1} | rows: 2514 | super-over innings: 34


Then conservation: every additive column's total in the innings table must equal the same total taken straight from the deliveries (a blank counts as nothing), so no delivery is lost or counted twice.

In [4]:
direct = {
    "runs_off_bat": int(deliveries["runs_off_bat"].sum()),
    "extras": int(deliveries["extras"].sum()),
    **{col: int(deliveries[col].sum()) for col in innings.EXTRAS_TYPES},
    "deliveries": len(deliveries),
    "legal_balls": int((deliveries["wides"].isna() & deliveries["noballs"].isna()).sum()),
    "wickets": int((deliveries["wicket_type"].notna() & ~deliveries["wicket_type"].isin(innings.NOT_WICKETS)).sum()),
}
conservation = pd.DataFrame({
    "innings table": innings_table[list(direct)].sum(),
    "straight from the deliveries": pd.Series(direct),
})
conservation["difference"] = conservation["innings table"] - conservation["straight from the deliveries"]
conservation

,innings table,straight from the deliveries,difference
runs_off_bat,381529,381529,0
extras,20209,20209,0
wides,11795,11795,0
noballs,1261,1261,0
byes,1389,1389,0
legbyes,5754,5754,0
penalty,10,10,0
deliveries,295732,295732,0
legal_balls,284630,284630,0
wickets,14686,14686,0


In [5]:
innings_table.to_csv(PROCESSED / "innings.csv", index=False)
loaded = tables.read_innings(PROCESSED / "innings.csv")
pd.testing.assert_frame_equal(loaded, innings_table)
print(f"saved {PROCESSED / 'innings.csv'}: {len(loaded):,} rows; read_innings() returns the in-memory table exactly, dtypes included")

saved ../data/processed/innings.csv: 2,514 rows; read_innings() returns the in-memory table exactly, dtypes included


**Reading Part B**
- **The grain is right.** 2,514 rows, one per (match, innings): no pair repeats, every innings has exactly one batting team, and the innings numbers run 1..k without gaps in all 1,243 matches — 1,221 matches with two innings, 6 with one, 15 with a super over (four innings) and 1 with two super overs (six).
- **Nothing was lost or counted twice.** Each of the ten additive columns has the same total in the innings table as taken straight from the 295,732 deliveries (difference 0 everywhere), including the two 5-run penalties (10 runs) found in notebook 01.
- **Super-over innings are flagged, not dropped:** 34 rows have `is_super_over = True`. The result checks below use innings 1 and 2 only.
- The table round-trips through CSV exactly (`read_innings`, dtypes included — the `is_super_over` boolean among them), so later notebooks can load it without re-deriving anything.

## Part C — Reconcile with `matches`

`matches` holds the *recorded* results (winner, margins, target); the innings table holds what the deliveries add up to. Two independent sources should agree. The helper table below puts them side by side, with the regular innings (1 and 2) only — super-over innings stay out of these totals.

In [6]:
regular = innings_table[~innings_table["is_super_over"]]
first = regular[regular["innings"] == 1].set_index("match_id")[["batting_team", "runs", "wickets"]].add_prefix("i1_")
second = regular[regular["innings"] == 2].set_index("match_id")[["batting_team", "runs", "wickets"]].add_prefix("i2_")
m = matches.join(first, on="match_id").join(second, on="match_id")          # one row per match

m["higher_total_team"] = np.where(m["i1_runs"] > m["i2_runs"], m["i1_batting_team"],
                                  np.where(m["i2_runs"] > m["i1_runs"], m["i2_batting_team"], "level"))
m["run_difference"] = (m["i1_runs"] - m["i2_runs"]).abs()
m["chaser_won"] = m["winner"] == m["i2_batting_team"]
m[["match_id", "season", "winner", "winner_runs", "winner_wickets", "method", "i1_runs", "i2_runs", "i2_wickets", "higher_total_team"]].head(4)

,match_id,season,winner,winner_runs,winner_wickets,method,i1_runs,i2_runs,i2_wickets,higher_total_team
0,335982,2007/08,Kolkata Knight Riders,140,<NA>,NaN,222,82.0,10.0,Kolkata Knight Riders
1,335983,2007/08,Chennai Super Kings,33,<NA>,NaN,240,207.0,4.0,Chennai Super Kings
2,335984,2007/08,Delhi Daredevils,<NA>,9,NaN,129,132.0,1.0,Delhi Daredevils
3,335985,2007/08,Royal Challengers Bangalore,<NA>,5,NaN,165,166.0,5.0,Royal Challengers Bangalore


### C1 · Wickets: which dismissals count?

Notebook 01 left this open: `retired hurt`, `retired out`, `obstructing the field` and `run out` are not the bowler's wickets, but are they wickets *lost by the batting side*? The recorded margin of a chase won by wickets — "won by N wickets" — is an independent answer: it must equal 10 minus the wickets the chasing team lost. So build each candidate definition side by side and let the 650 plain chases won by wickets judge them.

In [7]:
dismissal_types = sorted(deliveries["wicket_type"].dropna().unique())
bowler_types = ["bowled", "caught", "caught and bowled", "lbw", "stumped", "hit wicket"]
candidates = {
    "A every dismissal": dismissal_types,
    "B all but retired hurt": [t for t in dismissal_types if t != "retired hurt"],
    "C all but retired hurt, retired out": [t for t in dismissal_types if t not in ("retired hurt", "retired out")],
    "D all but retired hurt, obstructing the field": [t for t in dismissal_types if t not in ("retired hurt", "obstructing the field")],
    "E bowler's wickets only": bowler_types,
    "F bowler's wickets + run out": bowler_types + ["run out"],
}

chases = m[m["winner_wickets"].notna() & m["method"].isna()].set_index("match_id")       # plain chases won by wickets
second_innings = deliveries[deliveries["innings"] == 2]
lost = pd.DataFrame({
    name: second_innings["wicket_type"].isin(types).groupby(second_innings["match_id"]).sum()
    for name, types in candidates.items()
}).reindex(chases.index)
broken = lost.rsub(10).ne(chases["winner_wickets"], axis=0)                              # 10 - wickets lost != recorded margin

pd.DataFrame({
    "dismissal rows it counts (all innings)": {name: int(deliveries["wicket_type"].isin(types).sum()) for name, types in candidates.items()},
    f"recorded margins it breaks (of {len(chases)})": broken.sum(),
})

,dismissal rows it counts (all innings),recorded margins it breaks (of 650)
A every dismissal,14705,7
B all but retired hurt,14686,0
"C all but retired hurt, retired out",14680,0
"D all but retired hurt, obstructing the field",14682,2
E bowler's wickets only,13482,138
F bowler's wickets + run out,14676,2


In [8]:
# The rows that break the naive count (A): every one has a `retired hurt` in the chasing innings.
naive = broken["A every dismissal"]
retired_hurt_by_match = second_innings[second_innings["wicket_type"] == "retired hurt"].groupby("match_id").size()
pd.DataFrame({
    "wickets in hand, recorded": chases.loc[naive, "winner_wickets"],
    "10 - every dismissal": 10 - lost.loc[naive, "A every dismissal"],
    "10 - all but retired hurt": 10 - lost.loc[naive, "B all but retired hurt"],
    "retired hurt in innings 2": retired_hurt_by_match.reindex(chases.index[naive]),
}).join(chases.loc[naive, ["season", "winner"]])

,"wickets in hand, recorded",10 - every dismissal,10 - all but retired hurt,retired hurt in innings 2,season,winner
match_id,,,,,,
548306,8,7,8,1,2012,Mumbai Indians
598060,7,6,7,1,2013,Mumbai Indians
1082628,5,4,5,1,2017,Mumbai Indians
1136589,6,5,6,1,2018,Kolkata Knight Riders
1304067,8,7,8,1,2022,Sunrisers Hyderabad
1473499,6,5,6,1,2025,Sunrisers Hyderabad
1529269,6,5,6,1,2026,Delhi Capitals


In [9]:
# And the rows that break when `obstructing the field` is left out (D): counting it is what reproduces the margin.
obstructing = broken["D all but retired hurt, obstructing the field"]
obstructing_by_match = second_innings[second_innings["wicket_type"] == "obstructing the field"].groupby("match_id").size()
pd.DataFrame({
    "wickets in hand, recorded": chases.loc[obstructing, "winner_wickets"],
    "10 - counting it": 10 - lost.loc[obstructing, "B all but retired hurt"],
    "10 - leaving it out": 10 - lost.loc[obstructing, "D all but retired hurt, obstructing the field"],
    "obstructing the field in innings 2": obstructing_by_match.reindex(chases.index[obstructing]),
}).join(chases.loc[obstructing, ["season", "winner"]])

,"wickets in hand, recorded",10 - counting it,10 - leaving it out,obstructing the field in innings 2,season,winner
match_id,,,,,,
1181766,2,2,3,1,2019,Delhi Capitals
1426299,5,5,6,1,2024,Chennai Super Kings


In [10]:
# Where the data are silent: `retired out`. Does any innings with a retired-out batter carry a wicket margin?
retired_out = deliveries[deliveries["wicket_type"] == "retired out"][["match_id", "innings"]].drop_duplicates()
retired_out = retired_out.merge(m[["match_id", "season", "winner", "winner_wickets"]], on="match_id")
retired_out["innings carries a wicket margin"] = (retired_out["innings"] == 2) & retired_out["winner_wickets"].notna()
retired_out

,match_id,innings,season,winner,winner_wickets,innings carries a wicket margin
0,1304066,1,2022,Rajasthan Royals,<NA>,False
1,1359538,2,2023,Delhi Capitals,<NA>,False
2,1370352,1,2023,Gujarat Titans,<NA>,False
3,1473453,2,2025,Lucknow Super Giants,<NA>,False
4,1473459,2,2025,Punjab Kings,<NA>,False
5,1527691,1,2026,Chennai Super Kings,<NA>,False


In [11]:
passing = broken.sum()[lambda s: s == 0].index.tolist()
print("definitions that reproduce every recorded wicket margin:", passing)
assert [t for t in dismissal_types if t not in innings.NOT_WICKETS] == candidates["B all but retired hurt"]
print("innings.NOT_WICKETS =", innings.NOT_WICKETS, "-> definition B, one of the two that pass")

definitions that reproduce every recorded wicket margin: ['B all but retired hurt', 'C all but retired hurt, retired out']
innings.NOT_WICKETS = ['retired hurt'] -> definition B, one of the two that pass


**Reading C1**
- **Counting every dismissal (A) is wrong for the batting side's wickets.** It breaks 7 of the 650 recorded margins, and in each of the 7 the chasing innings contains a `retired hurt` batter — counting him gives one wicket too many. Leaving `retired hurt` out (B) reproduces all 650.
- **`obstructing the field` counts.** Leaving it out (D) breaks two margins (1181766 and 1426299), each with one such dismissal in the chase; counting it reproduces both.
- **Bowler-credited wickets only (E) answers a different question.** It breaks 138 margins because a run out is a wicket lost too; adding run outs back (F) leaves only the two obstructing-the-field cases.
- **`retired out` is not settled by the data.** B (counts it) and C (does not) both reproduce all 650 margins, because all six `retired out` dismissals are in an innings 1 or in a losing chase — none carries a wicket margin (table above). The definition counts it because the Laws treat it as a dismissal; that is outside knowledge, not evidence from this archive, and it moves the wickets of exactly six innings by one each.
- **Decision, now a named constant:** `innings.NOT_WICKETS = ["retired hurt"]` — a wicket lost is any dismissal except `retired hurt`. The cell above asserts that the constant is definition B, one of the two that survive the evidence.

### C2 · D/L matches: deriving the margins

Notebook 02 typed in two "explained" counts (14 and 13) for the 23 D/L matches. Here the formulas are derived and every figure computed. D/L replaces the target, so the margin must be measured against `target_runs`, not against the first-innings total. Let `par = target_runs - 1`, the score that levels the revised target.

In [12]:
dl = m[m["method"].notna()].copy()
dl["par"] = dl["target_runs"] - 1
dl["chaser_minus_par"] = dl["i2_runs"] - dl["par"]

by_runs = dl[dl["winner_runs"].notna()].copy()
by_runs["formula |chaser - par|"] = by_runs["chaser_minus_par"].abs()
by_wickets = dl[dl["winner_wickets"].notna()]

print(f"wins by runs ({len(by_runs)}): |chaser total - par| equals the recorded margin in {int((by_runs['formula |chaser - par|'] == by_runs['winner_runs']).sum())}")
print(f"  ...the chasing team won exactly when it was ahead of par: {int(((by_runs['chaser_minus_par'] > 0) == by_runs['chaser_won']).sum())} of {len(by_runs)}")
print(f"wins by wickets ({len(by_wickets)}): 10 - wickets lost by the chasers equals the recorded margin in {int((10 - by_wickets['i2_wickets'] == by_wickets['winner_wickets']).sum())}")
print(f"all D/L results ({len(dl)}): chaser total >= target_runs predicts the winner in {int(((dl['i2_runs'] >= dl['target_runs']) == dl['chaser_won']).sum())}")
by_runs[["season", "winner", "target_overs", "target_runs", "par", "i2_runs", "chaser_minus_par", "winner_runs", "chaser_won"]]

wins by runs (13): |chaser total - par| equals the recorded margin in 13
  ...the chasing team won exactly when it was ahead of par: 13 of 13
wins by wickets (10): 10 - wickets lost by the chasers equals the recorded margin in 10
all D/L results (23): chaser total >= target_runs predicts the winner in 23


,season,winner,target_overs,target_runs,par,i2_runs,chaser_minus_par,winner_runs,chaser_won
40,2007/08,Kings XI Punjab,8,89,88,94.0,6.0,6,True
43,2007/08,Chennai Super Kings,8,53,52,55.0,3.0,3,True
63,2009,Kolkata Knight Riders,9.2,69,68,79.0,11.0,11,True
89,2009,Chennai Super Kings,18,187,186,174.0,-12.0,12,False
221,2011,Kolkata Knight Riders,10,52,51,61.0,10.0,10,True
477,2015,Sunrisers Hyderabad,12,118,117,101.0,-16.0,16,False
538,2016,Rising Pune Supergiants,11,61,60,94.0,34.0,34,True
565,2016,Rising Pune Supergiants,11,58,57,76.0,19.0,19,True
566,2016,Royal Challengers Bangalore,14,203,202,120.0,-82.0,82,False
641,2018,Rajasthan Royals,6,71,70,60.0,-10.0,10,False


**Reading C2**
- **Win by runs (13 matches): margin = |chaser total − par|, with par = `target_runs` − 1.** If the chasing team finished *behind* par, the first-batting team wins by the shortfall; if it was *ahead* of par when play ended it wins by the surplus. One formula, 13 of 13 — and 6 of the 13 "wins by runs" are chasing-team wins, which is why a raw first-innings-minus-second-innings difference cannot reproduce them.
- **Win by wickets (10 matches): 10 − wickets lost by the chasers**, exactly as in a plain chase (10 of 10).
- **Winner (23 matches): the chasing team wins exactly when its total reaches `target_runs`** (23 of 23).
- So the 14 "winner has the lower raw total" and 13 "margin ≠ raw difference" figures that notebook 02 typed in are now computed: in the final table each of those rows also satisfies the formula above.

### C3 · Super overs: is the eliminator the higher total?

For each of the 16 super-over matches take the *final* super-over pair (innings 3 and 4, or 5 and 6 in the match that needed two) and compare the totals with the `eliminator` — the recorded super-over winner.

In [13]:
so = innings_table[innings_table["is_super_over"]].sort_values(["match_id", "innings"])
last_two = so.groupby("match_id").tail(2).assign(side=lambda d: d.groupby("match_id").cumcount())
fields = ["innings", "batting_team", "runs", "wickets"]
pair = (last_two[last_two["side"] == 0].set_index("match_id")[fields].add_suffix("_0")
        .join(last_two[last_two["side"] == 1].set_index("match_id")[fields].add_suffix("_1"))
        .join(matches.set_index("match_id")[["season", "eliminator"]]))

pair["higher_total_team"] = np.where(pair["runs_0"] > pair["runs_1"], pair["batting_team_0"],
                                     np.where(pair["runs_1"] > pair["runs_0"], pair["batting_team_1"], "level"))
pair["fewer_wickets_team"] = np.where(pair["wickets_0"] < pair["wickets_1"], pair["batting_team_0"],
                                      np.where(pair["wickets_1"] < pair["wickets_0"], pair["batting_team_1"], "level"))
pair["agrees"] = pair["higher_total_team"] == pair["eliminator"]
print(f"final super-over pair: the eliminator is the team with the higher total in {int(pair['agrees'].sum())} of {len(pair)} matches")
pair[["season", "innings_0", "batting_team_0", "runs_0", "wickets_0", "batting_team_1", "runs_1", "wickets_1", "higher_total_team", "eliminator", "agrees"]]

final super-over pair: the eliminator is the team with the higher total in 15 of 16 matches


,season,innings_0,batting_team_0,runs_0,wickets_0,batting_team_1,runs_1,wickets_1,higher_total_team,eliminator,agrees
match_id,,,,,,,,,,,
392190,2009,3,Kolkata Knight Riders,15,1,Rajasthan Royals,18,0,Rajasthan Royals,Rajasthan Royals,True
419121,2009/10,3,Chennai Super Kings,9,2,Kings XI Punjab,10,1,Kings XI Punjab,Kings XI Punjab,True
598004,2013,3,Sunrisers Hyderabad,20,0,Royal Challengers Bangalore,15,0,Sunrisers Hyderabad,Sunrisers Hyderabad,True
598017,2013,3,Royal Challengers Bangalore,15,0,Delhi Daredevils,11,2,Royal Challengers Bangalore,Royal Challengers Bangalore,True
729315,2014,3,Kolkata Knight Riders,11,2,Rajasthan Royals,11,0,level,Rajasthan Royals,False
829741,2015,3,Kings XI Punjab,15,2,Rajasthan Royals,6,2,Kings XI Punjab,Kings XI Punjab,True
1082625,2017,3,Mumbai Indians,11,2,Gujarat Lions,6,0,Mumbai Indians,Mumbai Indians,True
1175365,2019,3,Delhi Capitals,10,1,Kolkata Knight Riders,7,1,Delhi Capitals,Delhi Capitals,True
1178426,2019,3,Sunrisers Hyderabad,8,2,Mumbai Indians,9,0,Mumbai Indians,Mumbai Indians,True


In [14]:
# The exception, and the one match that needed two super overs: its first pair, which was level as well.
double_matches = so.groupby("match_id")["innings"].max().loc[lambda s: s > 4].index
print("exception(s):", pair.index[~pair["agrees"]].tolist(), "| matches with two super overs:", double_matches.tolist())
so[so["match_id"].isin(double_matches) & so["innings"].isin([3, 4])][["match_id", "innings", "batting_team", "runs", "wickets"]]

exception(s): [729315] | matches with two super overs: [1216517]


,match_id,innings,batting_team,runs,wickets
1584,1216517,3,Kings XI Punjab,5,2
1585,1216517,4,Mumbai Indians,5,1


**Reading C3**
- In **15 of 16** matches the eliminator is the team with the higher total in the final super-over pair.
- **The exception is 729315 (2014).** Both teams scored 11; KKR lost 2 wickets, RR none, and the eliminator is RR — the team that lost fewer wickets. The data are consistent with a fewer-wickets tie-break, but this is one case, and the data alone cannot confirm the rule.
- **The 2020 double super over argues against treating it as a general rule.** In 1216517 the first pair was level on runs too (5–5), with Mumbai losing 1 wicket and Kings XI 2, yet it was not settled on wickets: a second super over was played. So the tie-break differed between 2014 and 2020. The data show *that* it differed; *which* rule applied, and when it changed, needs the playing conditions of those seasons (outside knowledge).
- In practice: take super-over winners from `eliminator`, and do not recompute them from totals.

### C4 · The overs with a seventh legal delivery (notebook 01, hazard 3)

In [15]:
d = deliveries[["match_id", "innings", "ball", "runs_off_bat", "extras", "wides", "noballs", "wicket_type"]].copy()
d["over"] = np.floor(d["ball"]).astype(int)
d["legal"] = d["wides"].isna() & d["noballs"].isna()
d["legal_number"] = d["legal"].astype(int).groupby([d["match_id"], d["innings"], d["over"]]).cumsum().where(d["legal"])
seventh = d[d["legal_number"] == 7]
seven_ball_matches = sorted(seventh["match_id"].unique())
assert seven_ball_matches == [335994, 392198, 419155, 1136564]          # the four overs found in notebook 01

# Do these matches reconcile exactly? The same checks as the plain results in the final table, for just these four.
four = m[m["match_id"].isin(seven_ball_matches)].set_index("match_id")
four_checks = pd.DataFrame({
    "plain result": four["method"].isna() & four["winner"].notna(),
    "winner = higher total": four["winner"] == four["higher_total_team"],
    "target_runs = innings-1 total + 1": four["target_runs"] == four["i1_runs"] + 1,
    "margin reproduced": np.where(four["winner_runs"].notna(), four["winner_runs"] == four["run_difference"], four["winner_wickets"] == 10 - four["i2_wickets"]),
})
four_checks["reconciles exactly"] = four_checks.all(axis=1)
four_checks.join(seventh.set_index("match_id")[["innings", "ball", "runs_off_bat", "extras", "wicket_type"]].add_prefix("7th delivery: "))

,plain result,winner = higher total,target_runs = innings-1 total + 1,margin reproduced,reconciles exactly,7th delivery: innings,7th delivery: ball,7th delivery: runs_off_bat,7th delivery: extras,7th delivery: wicket_type
match_id,,,,,,,,,,
335994,True,True,True,True,True,2,10.8,0,0,NaN
392198,True,True,True,True,True,2,10.7,0,0,NaN
419155,True,True,True,True,True,1,18.9,0,0,NaN
1136564,True,True,True,True,True,2,11.7,1,0,NaN


In [16]:
# The innings that contain the seventh delivery, and how many legal balls they add up to.
innings_with_seventh = seventh[["match_id", "innings"]].merge(innings_table[["match_id", "innings", "legal_balls", "deliveries", "runs", "wickets"]], on=["match_id", "innings"])
innings_with_seventh

,match_id,innings,legal_balls,deliveries,runs,wickets
0,1136564,2,96,100,127,1
1,335994,2,73,77,155,0
2,392198,2,112,118,147,5
3,419155,1,121,126,112,9


**Reading C4**
- All four matches **reconcile exactly**: plain results, winner = higher total, `target_runs` = first-innings total + 1, margin reproduced.
- What that proves is limited. Three of the four seventh deliveries scored nothing and took no wicket, so they cannot move any total. The fourth (match 1136564) scored one run in a chase won by wickets, where even without that run Sunrisers (127 against a target of 126) still reach the target and the wicket margin is unchanged. So nothing contradicts the recorded results — but the reconciliation cannot show whether the seventh delivery was really bowled (an umpire miscount) or is a recording artefact.
- One more observation: in 419155 the innings with the seven-ball over is a full innings with **121 legal balls**, more than 20 overs' worth, so the extra delivery is certainly *recorded* in the data. Scorecards or umpire reports (outside knowledge) are needed to say more.

### C5 · One table: rule → violations → explained

Every rule in this notebook (and the scoreboard rules of notebook 02, now computed from the innings table). `violations` counts the matches or innings that break a rule. `explained` is computed: the violations that also satisfy the cause derived above. The cell **fails** if any violation is unexplained.

In [17]:
rules = []

def rule(check, name, broken, explained_by=None, why=""):
    """`broken`: a count or a boolean mask. `explained_by`: a boolean mask over the same rows - the cause derived in this notebook."""
    if np.isscalar(broken):
        count, explained = int(broken), 0
    else:
        broken = pd.Series(broken).fillna(False).astype(bool)
        count = int(broken.sum())
        explained = 0 if explained_by is None else int((broken & pd.Series(explained_by).fillna(False).astype(bool)).sum())
    rules.append({"check": check, "rule": name, "violations": count, "explained": explained, "why": why})

# --- grain of the innings table
rule("grain", "(match_id, innings) occurs more than once", innings_table.duplicated(["match_id", "innings"]).sum())
rule("grain", "an innings has more than one batting team", (per_innings["batting_team"].nunique() > 1).sum())
rule("grain", "innings numbers are not 1..k without gaps", (~((numbers["min"] == 1) & (numbers["max"] == numbers["nunique"]))).sum())
for col in conservation.index:
    rule("conservation", f"{col}: innings-table total != total from the deliveries", conservation.loc[col, "difference"] != 0)
rule("grain", "the batting teams of innings 1 and 2 are not team1 / team2", ~(((m["i1_batting_team"] == m["team1"]) & (m["i2_batting_team"] == m["team2"]))
                                                                               | ((m["i1_batting_team"] == m["team2"]) & (m["i2_batting_team"] == m["team1"]))) & m["i2_batting_team"].notna())

# --- plain results (no method)
plain = m[m["winner"].notna() & m["method"].isna()]
rule("plain results", "winner is not the team with the higher total", plain["winner"] != plain["higher_total_team"])
rule("plain results", "win by runs: winner_runs != run difference", plain["winner_runs"].notna() & (plain["winner_runs"] != plain["run_difference"]))
rule("plain results", "win by wickets: winner_wickets != 10 - wickets lost (chosen definition)", chases["winner_wickets"] != 10 - chases["i2_wickets"])
rule("plain results", "target_runs != innings-1 total + 1", plain["target_runs"] != plain["i1_runs"] + 1)
ties = m[m["outcome"] == "tie"]
rule("plain results", "tie: innings 1 and 2 totals differ", ties["i1_runs"] != ties["i2_runs"])
rule("wicket definition", "win by wickets: margin != 10 - EVERY dismissal (candidate A)", broken["A every dismissal"],
     explained_by=retired_hurt_by_match.reindex(chases.index).fillna(0) > 0, why="a retired-hurt batter is not a wicket")
rule("wicket definition", "win by wickets: margin != 10 - (all but obstructing; candidate D)", broken["D all but retired hurt, obstructing the field"],
     explained_by=obstructing_by_match.reindex(chases.index).fillna(0) > 0, why="obstructing the field is a wicket")

# --- D/L matches
rule("D/L", "winner is not the team with the higher raw total", dl["winner"] != dl["higher_total_team"],
     explained_by=(dl["i2_runs"] >= dl["target_runs"]) == dl["chaser_won"], why="the target was revised; the chaser reached it exactly when it won")
rule("D/L", "win by runs: winner_runs != raw run difference", by_runs["winner_runs"] != by_runs["run_difference"],
     explained_by=by_runs["winner_runs"] == by_runs["formula |chaser - par|"], why="margin = |chaser total - (target_runs - 1)|")
rule("D/L", "win by runs: winner_runs != |chaser total - (target_runs - 1)|", by_runs["winner_runs"] != by_runs["formula |chaser - par|"])
rule("D/L", "win by wickets: winner_wickets != 10 - wickets lost", by_wickets["winner_wickets"] != 10 - by_wickets["i2_wickets"])
rule("D/L", "'chaser total >= target_runs' disagrees with the winner", (dl["i2_runs"] >= dl["target_runs"]) != dl["chaser_won"])

# --- super overs and seven-ball overs
rule("super overs", "eliminator is not the higher total of the final super-over pair", ~pair["agrees"],
     explained_by=(pair["higher_total_team"] == "level") & (pair["fewer_wickets_team"] == pair["eliminator"]),
     why="level on runs; the eliminator lost fewer wickets (the rule itself needs outside knowledge)")
rule("seven-ball overs", "a match with a seventh legal delivery does not reconcile exactly", ~four_checks["reconciles exactly"])

validation = pd.DataFrame(rules)
unexplained = validation[validation["violations"] != validation["explained"]]
assert unexplained.empty, f"unexplained violations:\n{unexplained}"
validation

,check,rule,violations,explained,why
0,grain,"(match_id, innings) occurs more than once",0,0,
1,grain,an innings has more than one batting team,0,0,
2,grain,innings numbers are not 1..k without gaps,0,0,
3,conservation,runs_off_bat: innings-table total != total from the deli...,0,0,
4,conservation,extras: innings-table total != total from the deliveries,0,0,
5,conservation,wides: innings-table total != total from the deliveries,0,0,
6,conservation,noballs: innings-table total != total from the deliveries,0,0,
7,conservation,byes: innings-table total != total from the deliveries,0,0,
8,conservation,legbyes: innings-table total != total from the deliveries,0,0,
9,conservation,penalty: innings-table total != total from the deliveries,0,0,


## What this changes in notebook 01's worklist

| Item | Was | Now |
|---|---|---|
| 15 — which dismissals count as a wicket | "`run out`, `retired hurt`, `retired out`, `obstructing the field` are not bowler wickets; decide per question." | **Settled for wickets lost by the batting side:** every dismissal except `retired hurt` (`innings.NOT_WICKETS`). The recorded margins decide `retired hurt` (not counted: 7 margins) and `obstructing the field` (counted: 2 margins) and cannot decide `retired out` (counted per the Laws; its six cases carry no margin). **Still open:** a *bowler's* wickets (bowled, caught, caught and bowled, lbw, stumped, hit wicket) is a different question, and the innings table has no such column yet. |
| 16 — four overs with a seventh legal delivery | "Anomaly: flag; check against scorecards." | **Narrowed, not resolved.** All four matches reconcile exactly with their recorded results, but the seventh deliveries scored 0, 0, 0 and 1, so the check cannot tell whether they were really bowled. Keep the rows, keep the overs flagged; settling it needs outside sources. |
| 13 — super overs | "Flag; exclude from innings and match totals." | **Implemented:** `is_super_over` in the innings table, and innings 3+ stay out of every result check. The recorded winner is `eliminator`: totals reproduce it in 15 of 16 matches; the exception (2014) is a tie on runs where a fewer-wickets tie-break fits, which needs outside knowledge to confirm. |
| Notebook 02's typed "explained" counts (14 and 13) | Typed numbers. | **Computed** from the D/L formulas derived in C2. |

**Next:** the cleaning mappings (teams, venues, seasons, cities) — the first session where raw strings are allowed to change.